<a href="https://colab.research.google.com/github/kun1gund3/SDXL_GoogleCR-opt-v1.0/blob/main/SDXL_GoogleCR_opt_v1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎨 Stable Diffusion XL Base 1.0 — Google Colab Runner

**Model:** [`stabilityai/stable-diffusion-xl-base-1.0`](https://huggingface.co/stabilityai/stable-diffusion-xl-base-1.0)  

**Why SDXL Base 1.0?**
- ✅ Fits comfortably on free Colab T4 GPU (~7GB VRAM in bfloat16)
- 🏆 Industry-standard image generation model — well documented, widely used
- ⚡ Standard diffusers support — no custom classes, no hacks, just works
- 🔓 Open model license — free for personal and research use

**What this notebook gives you:**
- ✅ Section 0 — HuggingFace authentication
- ✅ Section 1 — Install dependencies & load the model
- ✅ Section 2 — Gradio UI with a **public shareable URL**
- ✅ Section 3 — FastAPI REST **API endpoint** (plug into any project)
- ✅ Section 4 — Code snippets for JS/Python integration

> ⚠️ **Enable GPU before running!**  
> `Runtime → Change runtime type → T4 GPU`

---
## 🔑 Section 0 — HuggingFace Token Setup

SDXL is a public model so the token is **optional** — but recommended for faster downloads and no rate-limiting warnings.

**Steps (one-time setup):**

1. Go to [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens)
2. Click **New token** → any name → permission: **Read** → click **Generate**
3. Copy the token (starts with `hf_...`)
4. In Colab, click the 🔑 **key icon** in the left sidebar (**Secrets**)
5. Click **Add new secret** → Name: `HF_TOKEN` → paste your token as Value
6. Toggle **Notebook access** ON

> ✅ Token is stored securely as a Colab secret — never visible in notebook code.

In [ ]:
# ─── 0.1  Authenticate with HuggingFace Hub ───────────────────────────────────
from google.colab import userdata
from huggingface_hub import login

try:
    hf_token = userdata.get("HF_TOKEN")
    login(token=hf_token, add_to_git_credential=False)
    print("✅ Logged in to HuggingFace Hub — faster downloads enabled!")
except Exception as e:
    print(f"⚠️  HF_TOKEN not found: {e}")
    print("   Continuing without token — model will still load (may be slower).")

---
## 🔧 Section 1 — Install & Load Model

**Why SDXL fits on free T4:**

| | SDXL Base | FLUX.1-schnell | Z-Image-Turbo |
|---|---|---|---|
| Parameters | ~3.5B | ~12B | ~6B |
| VRAM (bfloat16) | ~7 GB ✅ | ~24 GB ❌ | ~16 GB ❌ |
| Free T4 compatible | Yes | No | No |

SDXL uses a UNet architecture (older but proven) rather than a transformer — this makes it significantly lighter on VRAM than newer models.

In [ ]:
# ─── 1.1  Install dependencies ────────────────────────────────────────────────
!pip install -q \
    diffusers \
    transformers \
    accelerate \
    torch \
    gradio \
    fastapi \
    uvicorn \
    python-multipart \
    pyngrok \
    nest_asyncio

print("✅ Dependencies installed")

In [ ]:
# ─── 1.2  Load SDXL Base pipeline ─────────────────────────────────────────────
#
# WHY StableDiffusionXLPipeline?
#   SDXL's standard pipeline class in diffusers. Supports the full UNet-based
#   architecture with dual text encoders (CLIP ViT-L and OpenCLIP ViT-bigG).
#
# WHY torch_dtype=torch.float16?
#   float16 halves VRAM vs float32. We use float16 (not bfloat16) here because
#   T4 GPUs have better hardware support for float16, giving faster inference.
#
# WHY use_safetensors=True?
#   Loads weights from .safetensors format — faster and safer than .bin files.
#
# WHY variant='fp16'?
#   Downloads the pre-converted fp16 weight files directly from HuggingFace
#   instead of downloading fp32 and converting — saves ~3GB of download time.

import torch
from diffusers import StableDiffusionXLPipeline

MODEL_ID = "stabilityai/stable-diffusion-xl-base-1.0"

print("⏳ Loading SDXL Base 1.0...")
print("   First run downloads ~7GB of weights — takes 3–5 minutes.")
print("   Subsequent runs in the same session are instant (cached).")

pipe = StableDiffusionXLPipeline.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    use_safetensors=True,
    variant="fp16",
).to("cuda")

# Sliced attention reduces VRAM usage during generation with no quality loss
pipe.enable_attention_slicing()
pipe.set_progress_bar_config(disable=True)

print("\n✅ SDXL Base 1.0 loaded and ready!")
print(f"   VRAM used: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

In [ ]:
# ─── 1.3  Core inference function (shared by Gradio and FastAPI) ──────────────
import torch
from PIL import Image

def generate_image(
    prompt: str,
    negative_prompt: str = "blurry, low quality, distorted, watermark",
    width: int = 1024,
    height: int = 1024,
    steps: int = 30,
    guidance_scale: float = 7.5,
    seed: int = -1,
) -> tuple[Image.Image, int]:
    """
    Generate an image using SDXL Base 1.0.

    Args:
        prompt:          What you want in the image
        negative_prompt: What you don't want (artifacts, watermarks, etc.)
        width:           Image width  — SDXL is optimised for 1024x1024
        height:          Image height — SDXL is optimised for 1024x1024
        steps:           30 is the sweet spot for quality vs speed
        guidance_scale:  How closely to follow the prompt (7.5 is standard)
        seed:            -1 = random each time

    Returns:
        (PIL Image, seed used)
    """
    if seed == -1:
        seed = torch.randint(0, 2**32, (1,)).item()

    generator = torch.Generator(device="cuda").manual_seed(int(seed))

    result = pipe(
        prompt=prompt,
        negative_prompt=negative_prompt,
        width=width,
        height=height,
        num_inference_steps=steps,
        guidance_scale=guidance_scale,
        generator=generator,
    )

    return result.images[0], seed

# ─── Quick smoke test ─────────────────────────────────────────────────────────
print("🧪 Running quick test (512x512 to keep it fast)...")
test_img, test_seed = generate_image(
    prompt="A red apple on a wooden table, photorealistic",
    width=512, height=512, steps=20
)
test_img.save("/tmp/test.png")
print(f"✅ Test passed! Seed: {test_seed}")
print(f"   VRAM after generation: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

---
## 🌐 Section 2 — Gradio UI with Public URL

Launches a browser UI and gives you a `https://xxxx.gradio.live` public link.

In [ ]:
import gradio as gr

def gradio_generate(prompt, negative_prompt, width, height, steps, guidance_scale, seed):
    if not prompt.strip():
        return None, "⚠️ Please enter a prompt"
    image, used_seed = generate_image(
        prompt=prompt,
        negative_prompt=negative_prompt,
        width=int(width),
        height=int(height),
        steps=int(steps),
        guidance_scale=float(guidance_scale),
        seed=int(seed),
    )
    return image, f"✅ Done! Seed: {used_seed}  (save this to reproduce the image)"

with gr.Blocks(title="SDXL Base 1.0", theme=gr.themes.Soft()) as demo:
    gr.Markdown("""
    # 🎨 Stable Diffusion XL Base 1.0
    **High quality 1024x1024 image generation — running on free Colab T4 GPU**
    """)

    with gr.Row():
        with gr.Column(scale=1):
            prompt = gr.Textbox(
                label="Prompt",
                placeholder="A photorealistic portrait of an astronaut on Mars at sunset...",
                lines=3
            )
            negative_prompt = gr.Textbox(
                label="Negative Prompt (what to avoid)",
                value="blurry, low quality, distorted, watermark, ugly",
                lines=2
            )
            with gr.Row():
                width  = gr.Slider(512, 1024, value=1024, step=128, label="Width")
                height = gr.Slider(512, 1024, value=1024, step=128, label="Height")
            with gr.Row():
                steps          = gr.Slider(10, 50, value=30, step=1,   label="Steps")
                guidance_scale = gr.Slider(1.0, 15.0, value=7.5, step=0.5, label="Guidance Scale")
            seed = gr.Number(value=-1, label="Seed (-1 = random)")
            btn  = gr.Button("🎨 Generate Image", variant="primary", size="lg")

        with gr.Column(scale=1):
            output_image = gr.Image(label="Generated Image", type="pil")
            status       = gr.Textbox(label="Status", interactive=False)

    btn.click(
        fn=gradio_generate,
        inputs=[prompt, negative_prompt, width, height, steps, guidance_scale, seed],
        outputs=[output_image, status]
    )

    gr.Examples(
        examples=[
            ["A majestic snow leopard on a misty Himalayan peak, ultra-detailed fur, golden hour lighting, photorealistic", "blurry, low quality", 1024, 1024, 30, 7.5, 42],
            ["Aerial view of a tropical island at sunset, turquoise water, lush jungle, photorealistic drone shot", "blurry, low quality", 1024, 1024, 30, 7.5, -1],
            ["Close-up portrait of an elderly fisherman, weathered skin, piercing blue eyes, dramatic side lighting", "blurry, low quality, watermark", 1024, 1024, 30, 7.5, -1],
            ["Futuristic Tokyo street at night, neon reflections on wet pavement, cinematic", "blurry, low quality", 1024, 1024, 30, 7.5, -1],
        ],
        inputs=[prompt, negative_prompt, width, height, steps, guidance_scale, seed],
    )

demo.launch(
    share=True,
    debug=False,
    show_error=True,
)
# ⬆️ Look for: "Running on public URL: https://xxxx.gradio.live"

---
## 🚀 Section 3 — FastAPI REST Endpoint

Exposes SDXL as a REST API callable from any project.

> 🔑 **You need a free ngrok account.**  
> Get your token at [dashboard.ngrok.com/get-started/your-authtoken](https://dashboard.ngrok.com/get-started/your-authtoken)  
> Add it to Colab Secrets as `NGROK_TOKEN` (same steps as HF_TOKEN above).

In [ ]:
# ─── 3.1  Authenticate ngrok from Colab Secrets ───────────────────────────────
from google.colab import userdata

try:
    ngrok_token = userdata.get("NGROK_TOKEN")
    !ngrok authtoken {ngrok_token}
    print("✅ ngrok authenticated")
except Exception as e:
    print(f"❌ Could not load NGROK_TOKEN: {e}")
    print("   Add your ngrok token to Colab Secrets as NGROK_TOKEN")
    print("   Get it at: https://dashboard.ngrok.com/get-started/your-authtoken")

In [ ]:
# ─── 3.2  Build and launch FastAPI + ngrok tunnel ─────────────────────────────
import nest_asyncio
import uvicorn
import base64
from io import BytesIO
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel, Field
from pyngrok import ngrok

nest_asyncio.apply()

# ─── Request / Response schemas ───────────────────────────────────────────────
class GenerateRequest(BaseModel):
    prompt:          str   = Field(...,    description="Text prompt")
    negative_prompt: str   = Field("blurry, low quality, distorted, watermark", description="What to avoid")
    width:           int   = Field(1024,   ge=512, le=1024, description="Width (512 or 1024)")
    height:          int   = Field(1024,   ge=512, le=1024, description="Height (512 or 1024)")
    steps:           int   = Field(30,     ge=10,  le=50,   description="Inference steps")
    guidance_scale:  float = Field(7.5,    ge=1.0, le=15.0, description="Guidance scale")
    seed:            int   = Field(-1,     description="Seed (-1 = random)")

class GenerateResponse(BaseModel):
    image_base64: str = Field(..., description="Base64 PNG — use as: data:image/png;base64,{value}")
    seed_used:    int
    prompt:       str
    width:        int
    height:       int

# ─── FastAPI app ──────────────────────────────────────────────────────────────
app = FastAPI(
    title="SDXL Base 1.0 API",
    description="REST API for Stable Diffusion XL Base 1.0",
    version="1.0.0",
)
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])

@app.get("/")
def root():
    return {"model": "stabilityai/stable-diffusion-xl-base-1.0", "docs": "/docs"}

@app.get("/health")
def health():
    return {
        "status": "ok",
        "model": "stabilityai/stable-diffusion-xl-base-1.0",
        "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU",
        "vram_used_gb": round(torch.cuda.memory_allocated() / 1e9, 2),
        "vram_total_gb": round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2),
    }

@app.post("/generate", response_model=GenerateResponse)
def generate(req: GenerateRequest):
    """
    Generate an image from a text prompt.
    Returns a base64-encoded PNG.
    Use in HTML: <img src="data:image/png;base64,{image_base64}" />
    """
    try:
        image, seed_used = generate_image(
            prompt=req.prompt,
            negative_prompt=req.negative_prompt,
            width=req.width,
            height=req.height,
            steps=req.steps,
            guidance_scale=req.guidance_scale,
            seed=req.seed,
        )
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

    buffer = BytesIO()
    image.save(buffer, format="PNG")
    img_b64 = base64.b64encode(buffer.getvalue()).decode("utf-8")

    return GenerateResponse(
        image_base64=img_b64,
        seed_used=seed_used,
        prompt=req.prompt,
        width=req.width,
        height=req.height,
    )

# ─── Start ngrok tunnel ───────────────────────────────────────────────────────
public_tunnel = ngrok.connect(8000)
public_url    = public_tunnel.public_url

print("\n" + "="*60)
print("🚀 SDXL API is LIVE")
print("="*60)
print(f"  Public URL   : {public_url}")
print(f"  Swagger docs : {public_url}/docs")
print(f"  Health check : {public_url}/health")
print(f"  Generate     : POST {public_url}/generate")
print("="*60)
print()
print("📋 Quick test curl:")
print(f'  curl -X POST "{public_url}/generate" \\')
print( '       -H "Content-Type: application/json" \\')
print( '       -d \'{"prompt": "A red fox in a snowy forest, photorealistic"}\'')

# ─── Run server (async-compatible for Colab's existing event loop) ──────────
# uvicorn.run() creates its own event loop which crashes in Colab.
# uvicorn.Server + await plugs into Colab's existing asyncio loop instead.
import asyncio
nest_asyncio.apply()

config = uvicorn.Config(app, host="0.0.0.0", port=8000, loop="asyncio")
server = uvicorn.Server(config)
await server.serve()

---
## 📦 Section 4 — Calling the API from Your Project

Replace `YOUR_NGROK_URL` with the URL printed above.

---

### JavaScript / React
```javascript
const API_URL = "YOUR_NGROK_URL";

async function generateImage(prompt, options = {}) {
  const response = await fetch(`${API_URL}/generate`, {
    method: "POST",
    headers: { "Content-Type": "application/json" },
    body: JSON.stringify({
      prompt,
      negative_prompt: options.negativePrompt ?? "blurry, low quality, distorted",
      width: options.width ?? 1024,
      height: options.height ?? 1024,
      steps: options.steps ?? 30,
      guidance_scale: options.guidanceScale ?? 7.5,
      seed: options.seed ?? -1,
    }),
  });

  if (!response.ok) throw new Error(`API error: ${response.status}`);
  const data = await response.json();

  return {
    src: `data:image/png;base64,${data.image_base64}`,
    seed: data.seed_used,
  };
}

// Usage:
const { src, seed } = await generateImage("A sunset over the ocean, photorealistic");
// <img src={src} alt="generated" />
```

---

### Python
```python
import requests, base64
from PIL import Image
from io import BytesIO

API_URL = "YOUR_NGROK_URL"

response = requests.post(f"{API_URL}/generate", json={
    "prompt": "A futuristic city at night, neon lights, photorealistic",
    "negative_prompt": "blurry, low quality, watermark",
    "width": 1024,
    "height": 1024,
    "steps": 30,
    "guidance_scale": 7.5,
    "seed": -1,
})

data = response.json()
img = Image.open(BytesIO(base64.b64decode(data["image_base64"])))
img.save("output.png")
print(f"Saved! Seed used: {data['seed_used']}")
```

---

### API Reference

**POST `/generate`**

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `prompt` | string | required | What you want in the image |
| `negative_prompt` | string | `"blurry, low quality..."` | What to avoid |
| `width` | int | 1024 | 512 or 1024 |
| `height` | int | 1024 | 512 or 1024 |
| `steps` | int | 30 | 10–50 (more = better quality, slower) |
| `guidance_scale` | float | 7.5 | 1–15 (higher = follows prompt more strictly) |
| `seed` | int | -1 | -1 = random |

**Response:**
```json
{
  "image_base64": "iVBORw0KGgo...",
  "seed_used": 1234567890,
  "prompt": "your prompt here",
  "width": 1024,
  "height": 1024
}
```

**GET `/health`** — GPU status and VRAM usage  
**GET `/docs`** — Swagger UI to test endpoints in the browser